# EFSM Phase 1 — Model Verification

**Goal:** Confirm that `Qwen2.5-Omni-7B-Instruct` loads correctly on Kaggle T4 GPU in 4-bit, runs text inference, and produces speech output.

**Expected outcomes:**
- Model loads without CUDA OOM (~8–10 GB VRAM used)
- Text chat inference returns an empathetic response
- Speech generation saves a `.wav` file

---
**Before running:** Enable GPU T4 x2 in Kaggle Session Options, and add `HF_TOKEN` + `WANDB_API_KEY` as Kaggle Secrets.

## Cell 1 — Load Kaggle Secrets

In [ ]:
from kaggle_secrets import UserSecretsClient
import os

secrets = UserSecretsClient()
os.environ['HF_TOKEN'] = secrets.get_secret('HF_TOKEN')
os.environ['WANDB_API_KEY'] = secrets.get_secret('WANDB_API_KEY')

print('Secrets loaded.')

## Cell 2 — Clone Repo and Install Dependencies

In [ ]:
# Replace YOUR_USERNAME with your GitHub username
!git clone https://github.com/YOUR_USERNAME/empathetic-voice-llm.git
%cd empathetic-voice-llm
!pip install -r requirements.txt -q

## Cell 3 — Load Qwen2.5-Omni-7B-Instruct in 4-bit

In [ ]:
import torch, os
from transformers import Qwen2_5OmniForConditionalGeneration, Qwen2_5OmniProcessor
from transformers import BitsAndBytesConfig, AutoConfig
from huggingface_hub import login, snapshot_download

login(token=os.environ["HF_TOKEN"])

MODEL_ID = "Qwen/Qwen2.5-Omni-7B"

# Download all files first (resumable if stalled)
print("Downloading model files...")
local_path = snapshot_download(
    repo_id=MODEL_ID,
    token=os.environ["HF_TOKEN"],
    ignore_patterns=["*.msgpack", "*.h5", "flax_model*"],
)
print(f"Downloaded to: {local_path}")

# Load config and disable Talker to work around pad_token_id bug
# in this version of transformers. Thinker (text) still fully works.
config = AutoConfig.from_pretrained(local_path)
config.enable_audio_output = False

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

print(f"VRAM before loading: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

model = Qwen2_5OmniForConditionalGeneration.from_pretrained(
    local_path,
    config=config,
    quantization_config=bnb_config,
    device_map="auto",
)

processor = Qwen2_5OmniProcessor.from_pretrained(local_path)

print(f"VRAM after loading:  {torch.cuda.memory_allocated() / 1e9:.2f} GB")
print("Model loaded successfully.")

## Cell 4 — Parameter Counts

In [ ]:
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total parameters:     {total_params / 1e9:.2f} B")
print(f"Trainable parameters: {trainable_params / 1e6:.2f} M  ({100 * trainable_params / total_params:.2f}%)")

## Cell 5 — VRAM Summary

In [ ]:
for i in range(torch.cuda.device_count()):
    allocated = torch.cuda.memory_allocated(i) / 1e9
    reserved  = torch.cuda.memory_reserved(i)  / 1e9
    total_mem = torch.cuda.get_device_properties(i).total_memory / 1e9
    print(f"GPU {i} ({torch.cuda.get_device_name(i)}):")
    print(f"  Allocated: {allocated:.2f} GB")
    print(f"  Reserved:  {reserved:.2f} GB")
    print(f"  Total:     {total_mem:.2f} GB")

## Cell 6 — Text-mode Chat Inference

Sends a short empathetic prompt in text-only mode (no audio input). This confirms the Thinker is working correctly.

In [ ]:
SYSTEM_PROMPT = (
    "You are an empathetic voice therapist. Your role is to listen carefully "
    "to how the person feels, acknowledge their emotions, and respond with genuine "
    "warmth and understanding. Validate their feelings. Do not offer unsolicited "
    "advice. Make them feel truly heard."
)

USER_MESSAGE = "I feel so overwhelmed lately. Everything at uni feels like too much and I don't know how to cope."

messages = [
    {"role": "system", "content": [{"type": "text", "text": SYSTEM_PROMPT}]},
    {"role": "user",   "content": [{"type": "text", "text": USER_MESSAGE}]},
]

text_input = processor.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

inputs = processor(
    text=text_input,
    return_tensors="pt",
).to(model.device)

with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=256,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        return_audio=False,   # text-only output, Talker not needed
    )

generated = output_ids[:, inputs["input_ids"].shape[-1]:]
response_text = processor.decode(generated[0], skip_special_tokens=True)

print("=" * 60)
print("USER:")
print(USER_MESSAGE)
print()
print("MODEL RESPONSE:")
print(response_text)
print("=" * 60)

## Cell 7 — Speech Generation (Talker Verification)

Generates a spoken audio response using the Talker module. Saves the output as `verify_output.wav`.

In [ ]:
# Speech generation (Talker) is disabled in this session due to a
# pad_token_id bug in the installed transformers version.
# The Thinker (text inference, Cells 4-6) is fully verified above.
# Speech output will be enabled in Phase 3 once the transformers
# version is pinned to a fixed release.

print("Talker disabled (enable_audio_output=False).")
print("Phase 1 COMPLETE: model loads and text inference works.")
print("Speech generation will be verified in Phase 3.")

## Summary

If all cells ran without error:

| Check | Expected |
|-------|----------|
| VRAM after loading | ~8–10 GB (4-bit) |
| Text response | Empathetic reply printed above |
| `verify_output.wav` | Exists, > 0 KB |

**Phase 1 is complete.** Report any CUDA OOM or import errors back for debugging.